# Parameters

In [1]:
from pathlib import Path

import numpy as np

from simplex_qp import load_problem_config, partition_from_config, save_partition_metadata

config = load_problem_config()
np.random.seed(config.seed)

n = config.n
k = config.k
m = config.block_size
sizes = list(config.block_sizes)

load_path = config.data_folder
if not load_path.exists():
    raise FileNotFoundError(
        f"Folder {load_path} does not exist. Please run generate_matrices.ipynb first."
    )
config.results_folder.mkdir(parents=True, exist_ok=True)
config.summaries_folder.mkdir(parents=True, exist_ok=True)
load_folder = str(load_path) + "/"

partition = partition_from_config(config)
save_partition_metadata(
    load_path,
    partition,
    extra={"source": "problem_config", "problem_config": "config/problem_config.json"},
)

print(f"n: {n}, k: {k}, m: {m}, sizes: {sizes}")
print(f"Folder: {load_folder}")


n: 1000, k: 100, m: 10, sizes: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Folder: private/dim_n1000_k100/


# Generate x_u for q

### Scenario 1: solution inside boundaries

In [2]:
x_u_sc1 = np.zeros(n)
start = 0

for i, size in enumerate(sizes):
    # Genera valori strettamente positivi che sommano a 1 per il blocco corrente
    blocco = np.random.dirichlet(np.ones(size))
    x_u_sc1[start:start+size] = blocco
    
    print(f"Blocco {i+1} ({size} var): {np.round(blocco, 3)} | Somma: {np.sum(blocco):.2f}")
    start += size

print(f"\nx_u_sc1 finale: {np.round(x_u_sc1, 3)}")

Blocco 1 (10 var): [0.046 0.293 0.128 0.089 0.017 0.017 0.006 0.196 0.089 0.12 ] | Somma: 1.00
Blocco 2 (10 var): [0.003 0.44  0.224 0.03  0.025 0.025 0.046 0.093 0.071 0.043] | Somma: 1.00
Blocco 3 (10 var): [0.159 0.025 0.058 0.077 0.103 0.259 0.038 0.122 0.151 0.008] | Somma: 1.00
Blocco 4 (10 var): [0.082 0.016 0.006 0.261 0.296 0.145 0.032 0.009 0.101 0.051] | Somma: 1.00
Blocco 5 (10 var): [0.019 0.101 0.005 0.356 0.044 0.161 0.055 0.109 0.117 0.03 ] | Somma: 1.00
Blocco 6 (10 var): [0.245 0.105 0.197 0.158 0.064 0.179 0.007 0.015 0.003 0.028] | Somma: 1.00
Blocco 7 (10 var): [0.048 0.031 0.171 0.043 0.032 0.076 0.015 0.157 0.008 0.42 ] | Somma: 1.00
Blocco 8 (10 var): [0.184 0.028 0.001 0.21  0.152 0.162 0.183 0.01  0.055 0.015] | Somma: 1.00
Blocco 9 (10 var): [0.213 0.105 0.043 0.007 0.04  0.042 0.14  0.109 0.234 0.068] | Somma: 1.00
Blocco 10 (10 var): [0.018 0.173 0.198 0.114 0.204 0.094 0.102 0.077 0.004 0.016] | Somma: 1.00
Blocco 11 (10 var): [0.005 0.143 0.053 0.1   0.33

### Scenario 2: solution very outside boundaries

In [3]:
x_u_sc2 = np.zeros(n)
start = 0

for i, size in enumerate(sizes):
    if size == 1:
        x_u_sc2[start] = 1.0 # Se un blocco ha 1 sola variabile, deve per forza essere 1
    else:
        # Genera numeri con varianza estrema (es. tra -20 e +20)
        blocco = np.random.uniform(-20, 20, size)
        
        # Correzione: calcola quanto la somma differisce da 1 e trasla tutti i valori
        errore = np.sum(blocco) - 1.0
        blocco = blocco - (errore / size)
        
        x_u_sc2[start:start+size] = blocco
    
    print(f"Blocco {i+1} ({size} var): {np.round(x_u_sc2[start:start+size], 3)} | Somma: {np.sum(x_u_sc2[start:start+size]):.2f}")
    start += size

print(f"\nx_u_sc2 finale: {np.round(x_u_sc2, 3)}")

Blocco 1 (10 var): [-16.807  -2.536  10.705   5.077   8.05    2.139   3.479   9.755 -14.226
  -4.635] | Somma: 1.00
Blocco 2 (10 var): [-13.136  17.522  15.778 -20.407   6.238  15.025 -14.762   0.733  14.635
 -20.627] | Somma: 1.00
Blocco 3 (10 var): [ -0.656 -16.659   8.423  10.289   9.217  -9.585   5.928   5.229 -15.789
   4.603] | Somma: 1.00
Blocco 4 (10 var): [-11.229  11.141  -3.509  -7.887  -9.631  15.142   0.885  16.281 -10.095
  -0.098] | Somma: 1.00
Blocco 5 (10 var): [  1.978  12.027  -6.852   5.434  15.837   5.033 -10.283 -18.626  15.202
 -18.751] | Somma: 1.00
Blocco 6 (10 var): [  8.022  -5.809  10.596   4.985  12.951 -12.938   3.721 -10.889  -7.772
  -1.866] | Somma: 1.00
Blocco 7 (10 var): [ 14.272  18.688  10.056  -3.965  -3.821   8.828 -11.124 -16.256  -6.491
  -9.186] | Somma: 1.00
Blocco 8 (10 var): [ -4.998  -7.506 -15.167 -16.135  22.659   0.261  -1.477  10.336  -8.12
  21.148] | Somma: 1.00
Blocco 9 (10 var): [  9.202 -18.675  -5.548  12.913  15.538  -3.555   2.2

### Scenario 3: solution just a bit outside boundaries

In [4]:
x_u_sc3 = np.zeros(n)
start = 0

for i, size in enumerate(sizes):
    blocco = np.zeros(size)
    if size == 1:
        blocco[0] = 1.0 # Non si può violare se c'è 1 sola variabile
    else:
        # 1. Scegli a caso UNA SOLA variabile da rendere leggermente negativa
        idx_neg = np.random.randint(size)
        valore_negativo = -np.random.uniform(0.01, 0.1) # es. -0.05
        blocco[idx_neg] = valore_negativo
        
        # 2. La somma degli altri deve compensare per arrivare a 1
        somma_rimanente = 1.0 - valore_negativo
        
        # 3. Distribuisci la parte positiva rimanente tra gli altri elementi
        indici_restanti = [j for j in range(size) if j != idx_neg]
        blocco[indici_restanti] = np.random.dirichlet(np.ones(size - 1)) * somma_rimanente

    x_u_sc3[start:start+size] = blocco
    print(f"Blocco {i+1} ({size} var): {np.round(x_u_sc3[start:start+size], 3)} | Somma: {np.sum(x_u_sc3[start:start+size]):.2f}")
    start += size

print(f"\nx_u_sc3 finale: {np.round(x_u_sc3, 3)}")

Blocco 1 (10 var): [ 0.336  0.041  0.045  0.202  0.085  0.213 -0.032  0.01   0.095  0.005] | Somma: 1.00
Blocco 2 (10 var): [ 0.154  0.079  0.071 -0.066  0.275  0.017  0.05   0.246  0.148  0.026] | Somma: 1.00
Blocco 3 (10 var): [ 0.168  0.148  0.094  0.065  0.372  0.015 -0.045  0.123  0.02   0.039] | Somma: 1.00
Blocco 4 (10 var): [ 0.182  0.083 -0.097  0.023  0.032  0.049  0.056  0.368  0.16   0.144] | Somma: 1.00
Blocco 5 (10 var): [ 0.102 -0.046  0.204  0.104  0.124  0.033  0.244  0.073  0.02   0.142] | Somma: 1.00
Blocco 6 (10 var): [ 0.109  0.093  0.185  0.139  0.011  0.126  0.289  0.108 -0.07   0.009] | Somma: 1.00
Blocco 7 (10 var): [-0.028  0.032  0.019  0.043  0.153  0.079  0.025  0.227  0.229  0.221] | Somma: 1.00
Blocco 8 (10 var): [ 0.229  0.135  0.133  0.016  0.07   0.055  0.082  0.264 -0.06   0.077] | Somma: 1.00
Blocco 9 (10 var): [ 0.052  0.161  0.36   0.025  0.08   0.036 -0.084  0.17   0.093  0.108] | Somma: 1.00
Blocco 10 (10 var): [ 0.027  0.286  0.092  0.075  0.103

# Q_well

In [5]:
# load Q_well from npz file
Q_well = np.load(load_folder+"matrices.npz")["Q_well"]

q_well_sc1 = -2.0 * (Q_well @ x_u_sc1)
q_well_sc2 = -2.0 * (Q_well @ x_u_sc2)
q_well_sc3 = -2.0 * (Q_well @ x_u_sc3)

# Q_ill

In [6]:
# load Q_ill from npz file
Q_ill = np.load(load_folder+"matrices.npz")["Q_ill"]

q_ill_sc1 = -2.0 * (Q_ill @ x_u_sc1)
q_ill_sc2 = -2.0 * (Q_ill @ x_u_sc2)
q_ill_sc3 = -2.0 * (Q_ill @ x_u_sc3)

# Save generated data

In [7]:
# save q_well_sc1, q_well_sc2, q_well_sc3, q_ill_sc1, q_ill_sc2, q_ill_sc3 in npz file
np.savez(load_folder+"vectors.npz",
         q_well_sc1=q_well_sc1, q_well_sc2=q_well_sc2, q_well_sc3=q_well_sc3,
         q_ill_sc1=q_ill_sc1, q_ill_sc2=q_ill_sc2, q_ill_sc3=q_ill_sc3)

# save the unconstrained minimizers used to define the three scenarios
np.savez(load_folder+"targets.npz",
         x_u_sc1=x_u_sc1, x_u_sc2=x_u_sc2, x_u_sc3=x_u_sc3)